In [1]:
import sys, torch
sys.path.insert(0, "qwen3")                      # match the architecture of the checkpoint
from model import TinyQwen
from tokenizer import CharTokenizer


In [2]:

ckpt = torch.load("qwen3/tiny_qwen.pt", map_location="cpu", weights_only=False)
tok = CharTokenizer(ckpt["chars"])
model = TinyQwen(ckpt["cfg"]); model.load_state_dict(ckpt["model"]); model.eval()

model

TinyQwen(
  (embed_tokens): Embedding(30, 32)
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (input_layernorm): RMSNorm()
      (attn): Attention(
        (q_proj): Linear(in_features=32, out_features=32, bias=False)
        (k_proj): Linear(in_features=32, out_features=16, bias=False)
        (v_proj): Linear(in_features=32, out_features=16, bias=False)
        (o_proj): Linear(in_features=32, out_features=32, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=32, out_features=64, bias=False)
        (up_proj): Linear(in_features=32, out_features=64, bias=False)
        (down_proj): Linear(in_features=64, out_features=32, bias=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=32, out_features=30, bias=False)
)

In [4]:
start = torch.full((10, 1), tok.eos_id, dtype=torch.long)   # every name starts at EOS/newline
out = model.generate(start, max_new_tokens=model.cfg.max_seq_len,
                     temperature=0.8, eos_id=tok.eos_id)     # stops each row at EOS
out

tensor([[ 0,  8,  1, 23,  5, 12,  0,  0],
        [ 0, 18,  9, 12,  1, 14,  0,  0],
        [ 0,  1, 17, 20, 13,  1, 14,  0],
        [ 0, 14,  5, 23,  9,  8,  0,  0],
        [ 0,  9, 14, 14, 20, 17,  0,  0],
        [ 0, 18,  1, 12,  9,  8,  0,  0],
        [ 0, 23,  1, 17, 20, 14,  0,  0],
        [ 0, 21,  1,  8,  1, 19,  0,  0],
        [ 0,  7, 26, 12,  9, 23,  0,  0],
        [ 0,  3,  5, 14, 11,  0,  0,  0]])

In [5]:
for row in out.tolist():
    print(tok.decode(row[1:]).split("\n")[0])

hazel
silan
aruman
nezih
innur
salih
zarun
vahat
güliz
cenk
